# SA3 generation probe

Can Stable Audio 3 generate clean masters? Every released model type
generates the same track (one fixed prompt, seeds 0–2). Each clip is
MP3-compressed and the compression is scored against the clip itself — a
generated clip is its own reference. If MP3 removes *less* from a generation
than it removes from a real master (higher SDR, lower LSD cost), the
generation was missing the content MP3 normally eats: air, transient detail.

`scripts/run_probe.py` must have filled `artifacts/probe/` first (GPU pod).
The anchor rows are what MP3 removes from real masters, taken from the
benchmark's results.

**Colours: green = behaves most like the aerofunk master under the codec,
red = least.** Raw best/worst would praise the dullest clip — the easiest
track to compress wins SDR — so cells are ranked by distance to the anchor.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Audio, HTML, display

from grooveback import audio as ga

PROBE = Path("../artifacts/probe")
RESULTS = json.loads((PROBE / "results.json").read_text())
BITRATES = ("64k", "128k", "192k")
COLUMNS = ("BSS-SDR", "SDR", "SI-SNR", "Spectral SNR", "LSD ↓")
KEYS = ("bss_sdr_db", "sdr_db", "si_snr_db", "spectral_snr_db", "lsd_db")
ANCHOR = "aerofunk master"

variants = [v for v, block in RESULTS["variants"].items() if "error" not in block]
failed = {v: block["error"] for v, block in RESULTS["variants"].items()
          if "error" in block}
print("prompt:", RESULTS["prompt"])
if failed:
    for v, e in failed.items():
        print(f"FAILED {v}: {e}")


def anchored_table(title, rows, anchor_values):
    """Green = closest to the anchor in that column, red = farthest."""
    marks = []
    for i in range(len(anchor_values)):
        gaps = {label: abs(v[i] - anchor_values[i]) for label, v in rows}
        marks.append((min(gaps.values()), max(gaps.values())))
    header = "".join(f"<th style='padding:2px 12px'>{c}</th>" for c in ("", *COLUMNS))
    body = ""
    for label, values in [*rows, (ANCHOR, anchor_values)]:
        cells = ""
        for i, value in enumerate(values):
            style = "padding:2px 12px;text-align:right"
            if label != ANCHOR:
                gap = abs(value - anchor_values[i])
                if gap == marks[i][0]:
                    style += ";color:#0a8a0a;font-weight:bold"
                elif gap == marks[i][1]:
                    style += ";color:#cc2222;font-weight:bold"
            cells += f"<td style='{style}'>{value:.1f}</td>"
        name = label if label != ANCHOR else f"<i>{label}</i>"
        body += f"<tr><td style='padding:2px 12px'>{name}</td>{cells}</tr>"
    display(HTML(f"<h4 style='margin:10px 0 2px'>{title}</h4>"
                 f"<table><tr>{header}</tr>{body}</table>"))

In [ ]:
for bitrate in BITRATES:
    rows = [(v, [RESULTS["variants"][v][bitrate]["mean"][k] for k in KEYS])
            for v in variants]
    anchor = [RESULTS["anchors"][ANCHOR][bitrate][k] for k in KEYS]
    anchored_table(f"what MP3 @ {bitrate} removes — generations vs the master",
                   rows, anchor)

## Spectrograms

Seed 0 of each variant, next to its 64k round-trip.

In [ ]:
def spectrogram(ax, wav, title):
    audio, sr = ga.load(wav)
    ax.imshow(ga.spectrogram_db(audio), origin="lower", aspect="auto",
              cmap="magma", vmin=-100, vmax=0,
              extent=[0, audio.shape[1] / sr, 0, sr / 2 / 1000])
    ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(len(variants), 2, figsize=(13, 2.6 * len(variants)),
                         constrained_layout=True, sharex=True, sharey=True)
for row, variant in zip(axes if len(variants) > 1 else [axes], variants):
    spectrogram(row[0], PROBE / variant / "seed0.wav", variant)
    spectrogram(row[1], PROBE / variant / "64k" / "seed0.wav", f"{variant} @ 64k")
plt.show()

## Listen

45 s per variant, level-matched, with its 64k round-trip.

In [ ]:
for variant in variants:
    print(f"── {variant}")
    for wav in sorted((PROBE / variant / "listen").glob("*.wav")):
        print(wav.stem)
        display(Audio(str(wav)))